# Prophet — Walmart Store Sales Forecasting


## 0 · Setup — attach `preprocessor` and `walmart_ts_common`, plus the data.

In [ ]:
import sys, glob, time
for p in glob.glob("/kaggle/usr/lib/*") + glob.glob("/kaggle/input/*"):
    if p not in sys.path:
        sys.path.append(p)
try:
    from prophet import Prophet
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "prophet", "-q"])
    from prophet import Prophet
import logging
logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
import numpy as np, pandas as pd
from joblib import Parallel, delayed
import walmart_ts_common as C
print("Prophet ready")

## MLflow / DagsHub logging


In [ ]:
!pip install dagshub "mlflow>=2.22.0" -q

DAGSHUB_OWNER = "nikaduri"                           # <-- your DagsHub username
DAGSHUB_REPO  = "ml-store-sales-forecasting"   # <-- your DagsHub repo (create once)
MLFLOW_EXPERIMENT = "Prophet_Training"

_token = None
try:                                                 # Kaggle Secrets (preferred)
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
except Exception:
    _token = None                                    # -> dagshub OAuth fallback

C.setup_dagshub_mlflow(DAGSHUB_OWNER, DAGSHUB_REPO, token=_token)

## 1 · Panel + per-series frames

Same `build_panel()` (identical split & cleaning). Prophet wants `ds`/`y` per
series in **raw dollar units — no scaling** (it decomposes additively in the
original scale). `y` is still the cleaned/clipped training signal the other
models use — "raw units" means unstandardised, not un-cleaned.

In [ ]:
panel = C.build_panel()
store = C.SeriesStore(panel)
val_dates  = pd.to_datetime(store.val_dates)
test_dates = pd.to_datetime(store.test_dates)

train_rows = panel[panel["split"] == "train"][["Store", "Dept", "Date", "y"]]
series_frames = {
    key: g.sort_values("Date").rename(columns={"Date": "ds"})[["ds", "y"]].dropna()
    for key, g in train_rows.groupby(["Store", "Dept"], sort=False)
}
print("series:", len(store.series))

## 2 · Holidays — Prophet's native mechanism

Hand Prophet the four competition holidays via its own `holidays=` argument
(dates from `preprocessor.HOLIDAYS`), letting it learn a per-holiday effect. The
5× WMAE weighting is **not** baked in — that would fight the model; it stays at
evaluation only, in the shared scorer.

In [ ]:
from preprocessor import HOLIDAYS
holidays_df = pd.DataFrame([
    {"holiday": name, "ds": pd.Timestamp(d)}
    for name, dates in HOLIDAYS.items() for d in dates
])
holidays_df["holiday"].value_counts().to_dict()

## 3 · Fixed evaluation sample + the experiment harness

`EVAL_SAMPLE` is a fixed set of series used by **every** trial, so configs are
compared on identical series. `run_trial(cfg)` fits Prophet with `cfg`'s
hyperparameters on that sample (series below `min_history` fall back to
same-week-last-year), scores with the shared WMAE, and logs to the tracker.

Prophet knobs worth trialing: `seasonality_mode` (additive vs multiplicative —
retail sales are often multiplicative), `changepoint_prior_scale` (trend
flexibility), `holidays_prior_scale` (how hard holidays are fit), `yearly_order`
(seasonality richness), `min_history`.

Each trial logs the full metric family (`train_mae`, `val_mae`, `train_wmae`, `val_wmae`, `val_rmse`, `val_r2`); `train_*` is in-sample fit quality. Prophet isn't trained in epochs, so there's no per-epoch loss curve here — the per-config WMAE across trials is the analog.

In [ ]:
rng = np.random.default_rng(0)
val_keys = sorted(set(map(tuple, panel.loc[panel["split"] == "val", ["Store", "Dept"]]
                          .itertuples(index=False, name=None))) & set(store.series))
SAMPLE_SIZE = 300
idx = rng.permutation(len(val_keys))[:SAMPLE_SIZE]
EVAL_SAMPLE = [val_keys[i] for i in idx]
print(f"tuning on a fixed sample of {len(EVAL_SAMPLE)} series")

# holiday timestamps -> used to 5x-weight the in-sample (train) WMAE, matching val
HOL_DS = set(pd.to_datetime(holidays_df["ds"]).to_numpy())

def fit_one(key, frame, future_ds, cfg):
    # joblib workers are separate processes and don't inherit the
    # logging.getLogger(...).setLevel(...) calls made in cell 2 (main process
    # only) -- reapply here so each worker actually stays quiet too.
    import logging
    logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
    logging.getLogger("prophet").setLevel(logging.WARNING)
    m = Prophet(holidays=holidays_df, yearly_seasonality=cfg["yearly_order"],
                weekly_seasonality=False, daily_seasonality=False,
                seasonality_mode=cfg["seasonality_mode"],
                changepoint_prior_scale=cfg["changepoint_prior_scale"],
                holidays_prior_scale=cfg["holidays_prior_scale"])
    m.fit(frame)
    fc  = m.predict(pd.DataFrame({"ds": future_ds}))     # val-horizon forecast
    ins = m.predict(frame[["ds"]])                        # in-sample fit -> train metrics
    return (key, fc["yhat"].to_numpy(),
            frame["y"].to_numpy(), ins["yhat"].to_numpy(), frame["ds"].to_numpy())

TRIAL_CFGS = {}
def run_trial(cfg, tracker=None, note="", keys=None, subset_for_score=None,
              return_predictions=False):
    keys = EVAL_SAMPLE if keys is None else keys
    fit_keys   = [k for k in keys if k in series_frames
                  and series_frames[k]["y"].notna().sum() >= cfg["min_history"]]
    naive_keys = [k for k in keys if k not in fit_keys]
    t0 = time.time()
    results = Parallel(n_jobs=cfg["n_jobs"])(
        delayed(fit_one)(k, series_frames[k], val_dates, cfg) for k in fit_keys)
    secs = time.time() - t0

    pbk = {}
    tr_true, tr_pred, tr_ds = [], [], []
    for key, yhat_val, y_tr, yhat_tr, ds_tr in results:
        pbk[key] = yhat_val
        tr_true.append(y_tr); tr_pred.append(yhat_tr); tr_ds.append(ds_tr)
    for k in naive_keys:
        pbk[k] = store.naive_val(k)

    # ---- validation metrics (dollars, via the shared scorer) ----
    _, merged = C.score_val_predictions(panel, store, pbk,
                                        subset=(subset_for_score or keys))
    val_m = C.regression_metrics(merged["y_true"], merged["y_pred"],
                                 merged["IsHoliday"], prefix="val_")

    # ---- training metrics (in-sample fit across the fitted series) ----
    if tr_true:
        yt = np.concatenate(tr_true); yp = np.concatenate(tr_pred)
        hol = np.isin(np.concatenate(tr_ds), list(HOL_DS)).astype(float)
        train_m = C.regression_metrics(yt, yp, hol, prefix="train_")
    else:
        train_m = {"train_mae": float("nan"), "train_wmae": float("nan")}

    metrics = {
        "train_mae":  train_m.get("train_mae", float("nan")),
        "val_mae":    val_m["val_mae"],
        "train_wmae": train_m.get("train_wmae", float("nan")),
        "val_wmae":   val_m["val_wmae"],
        "val_rmse":   val_m["val_rmse"],
        "val_r2":     val_m["val_r2"],
        "n_fallback": len(naive_keys),
        "fit_secs":   round(secs, 1),
        "n_series":   len(keys),
    }
    # Prophet is not trained in epochs, so there is no per-epoch loss curve to
    # log (history=None); the analog is the per-config WMAE across trials.
    if tracker is not None:
        tid = tracker.log(cfg, metrics, note); TRIAL_CFGS[tid] = dict(cfg)
    if return_predictions:
        return metrics, merged
    return metrics

## 4 · Run the experiments (on the fixed sample)

One field changed per trial. WMAE here is on the sample, so it's for **relative**
comparison of configs — the comparable-to-other-models number comes from the full
run in §6.

In [ ]:
BASE = dict(seasonality_mode="additive", changepoint_prior_scale=0.05,
            holidays_prior_scale=10.0, yearly_order=10, min_history=104,
            n_jobs=4, seed=42)

tracker = C.ExperimentTracker("prophet", mlflow_experiment=MLFLOW_EXPERIMENT)
run_trial(BASE,                                           tracker, note="baseline")

# -- seasonality shape: retail sales are often multiplicative --
run_trial({**BASE, "seasonality_mode": "multiplicative"}, tracker, note="multiplicative")

# -- trend flexibility --
run_trial({**BASE, "changepoint_prior_scale": 0.5},       tracker, note="flexible trend")
run_trial({**BASE, "changepoint_prior_scale": 0.01},      tracker, note="stiffer trend")

# -- how hard to fit the four competition holidays (the 5x-weighted weeks) --
run_trial({**BASE, "holidays_prior_scale": 1.0},          tracker, note="weaker holidays")
run_trial({**BASE, "holidays_prior_scale": 20.0},         tracker, note="stronger holidays")

# -- seasonality richness --
run_trial({**BASE, "yearly_order": 20},                   tracker, note="richer seasonality")
run_trial({**BASE, "yearly_order": 6},                    tracker, note="smoother seasonality")

# -- eligibility: lower min_history fits more series natively (fewer naive fallbacks) --
run_trial({**BASE, "min_history": 60},                    tracker, note="fit more series")

## 5 · Compare trials

In [ ]:
display(tracker.summary())
tracker.diff(1, int(tracker.summary().iloc[0]["trial"]))
tracker.to_csv("/kaggle/working/prophet_experiments.csv")

### 5b · Trial comparison plot

Same `train_wmae` vs `val_wmae` per trial as the tree-model notebooks (`xg_trainvsval`) — Prophet has no per-epoch loss curve, so this per-config comparison is the direct analog. Configs where `val_wmae` sits far above `train_wmae` are overfitting the fixed sample; configs where both are high are underfitting.

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("/kaggle/working/images", exist_ok=True)
summary = tracker.summary().sort_values("val_wmae").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(summary))
width = 0.35
ax.bar(x - width/2, summary["train_wmae"], width, label="train_wmae", color="#9fc5e8")
ax.bar(x + width/2, summary["val_wmae"],   width, label="val_wmae",   color="#e06666")

best_x = 0  # summary is sorted ascending by val_wmae, so index 0 is the best trial
ax.axvspan(best_x - 0.5, best_x + 0.5, color="gold", alpha=0.15, zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(summary["note"], rotation=40, ha="right")
ax.set_ylabel("WMAE ($)")
ax.set_title("Prophet trials \u2014 train vs val WMAE by config")
ax.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/images/prophet_trainvsval.png", dpi=150)
plt.show()


## 6 · Full-population val score + submission (best config)

Now run the winning config on **all** eligible series for the number comparable
to the other four models, then refit on all labelled data and forecast the real
39 test weeks. This is the slow cell — run it once when you've settled on a config.

In [ ]:
RUN_FULL = True
FINAL_CFG = TRIAL_CFGS[int(tracker.summary().iloc[0]["trial"])]
print("full run + submission use best cfg:", FINAL_CFG)

if RUN_FULL:
    all_keys = list(store.series.keys())
    full_metrics, full_merged = run_trial(FINAL_CFG, tracker=None, note="full",
                             keys=all_keys, subset_for_score=None,
                             return_predictions=True)
    print("FULL val WMAE:", full_metrics["val_wmae"],
          "| fallback:", full_metrics["n_fallback"])

    # submission: refit each eligible series on train+val, forecast 39 test weeks
    full_rows = panel[panel["split"].isin(["train", "val"])][["Store", "Dept", "Date", "y"]]
    full_frames = {k: g.sort_values("Date").rename(columns={"Date": "ds"})[["ds", "y"]].dropna()
                   for k, g in full_rows.groupby(["Store", "Dept"], sort=False)}
    sub_fit = [k for k in full_frames if full_frames[k]["y"].notna().sum() >= FINAL_CFG["min_history"]]
    sub_res = Parallel(n_jobs=FINAL_CFG["n_jobs"])(
        delayed(fit_one)(k, full_frames[k], test_dates, FINAL_CFG) for k in sub_fit)
    sub_pred = {key: yhat_val for key, yhat_val, *_ in sub_res}
    for k in store.series:
        sub_pred.setdefault(k, store.naive_submit(k))
    submission = C.build_submission(panel, store, sub_pred)
    submission.to_csv("submission.csv", index=False)
    print("wrote submission.csv:", submission.shape)
    display(submission.head())


### 6b · Actual vs Predicted (full population)

Same idea as `xg_actualvspredicted` / `gbm_actualvspredicted`, built from `full_merged` — the per-series forecasts stacked across all eligible series from the full run above, so it's on the same footing as the pooled tree/deep models. Holiday weeks (5x-weighted) are called out separately since that's where WMAE is actually won or lost.

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("/kaggle/working/images", exist_ok=True)
fig, ax = plt.subplots(figsize=(7, 7))

is_hol = full_merged["IsHoliday"].astype(bool)
ax.scatter(full_merged.loc[~is_hol, "y_true"], full_merged.loc[~is_hol, "y_pred"],
           s=8, alpha=0.35, color="#6fa8dc", label="non-holiday")
ax.scatter(full_merged.loc[is_hol, "y_true"], full_merged.loc[is_hol, "y_pred"],
           s=14, alpha=0.6, color="#e06666", label="holiday (5x weight)")

lo = min(full_merged["y_true"].min(), full_merged["y_pred"].min())
hi = max(full_merged["y_true"].max(), full_merged["y_pred"].max())
ax.plot([lo, hi], [lo, hi], color="black", linewidth=1, linestyle="--", label="y = \u0177")

ax.set_xlabel("Actual sales ($)")
ax.set_ylabel("Predicted sales ($)")
ax.set_title(f"Prophet \u2014 actual vs predicted (full run, val_r2={full_metrics['val_r2']:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig("/kaggle/working/images/prophet_actualvspredicted.png", dpi=150)
plt.show()

resid = (full_merged["y_pred"] - full_merged["y_true"]).abs()
worst = full_merged.loc[[resid.idxmax()]]
print("largest residual:", round(resid.max(), 2), "-> row:")
display(worst)


### 6c · Log final artifacts (submission + experiment log + plots)

Moved here (after both plot cells) so the two PNGs actually exist on disk before
`log_artifacts_run` tries to upload them — doing this from inside §6 would have
logged a run missing the actual-vs-predicted plot, since that one isn't
generated until §6b.

In [ ]:
tracker.log_artifacts_run(
    "prophet_Best",
    params={**FINAL_CFG, "sample_size": SAMPLE_SIZE},
    metrics={"full_val_wmae":   float(full_metrics["val_wmae"]),
             "full_val_mae":    float(full_metrics["val_mae"]),
             "full_val_rmse":   float(full_metrics["val_rmse"]),
             "full_val_r2":     float(full_metrics["val_r2"]),
             "full_train_wmae": float(full_metrics["train_wmae"]),
             "full_train_mae":  float(full_metrics["train_mae"]),
             "n_fallback":      int(full_metrics["n_fallback"])},
    artifacts=["submission.csv", "/kaggle/working/prophet_experiments.csv",
               "/kaggle/working/images/prophet_trainvsval.png",
               "/kaggle/working/images/prophet_actualvspredicted.png"],
    texts={"holidays.txt": holidays_df.to_csv(index=False)})
